In [1]:
import numpy as np 
import pandas as pd
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../datasets/credit_risk_cleaned.csv')
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,cb_person_codes
0,21,9600,OWN,5.0,EDUCATION,1000,11.14,0,0.10,N,2,1
1,25,9600,MORTGAGE,1.0,MEDICAL,5500,12.87,1,0.57,N,3,1
2,23,65500,RENT,4.0,MEDICAL,35000,15.23,1,0.53,N,2,1
3,24,54400,RENT,8.0,MEDICAL,35000,14.27,1,0.55,Y,4,0
4,21,9900,OWN,2.0,VENTURE,2500,7.14,1,0.25,N,2,1


In [3]:
df.isna().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
cb_person_codes               0
dtype: int64

In [4]:
df.select_dtypes(include=['object']).nunique()

person_home_ownership        4
loan_intent                  6
cb_person_default_on_file    2
dtype: int64

In [5]:
X = df.drop(columns=['loan_status','loan_intent','cb_person_default_on_file','person_home_ownership'])
X.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,cb_person_codes
0,21,9600,5.0,1000,11.14,0.10,2,1
1,25,9600,1.0,5500,12.87,0.57,3,1
2,23,65500,4.0,35000,15.23,0.53,2,1
3,24,54400,8.0,35000,14.27,0.55,4,0
4,21,9900,2.0,2500,7.14,0.25,2,1


In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled_df.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,cb_person_codes
0,-1.090587,-1.078051,0.054432,-1.367192,0.034115,-0.655113,-0.939656,0.464870
1,-0.441211,-1.078051,-0.938456,-0.656810,0.597575,3.767461,-0.692664,0.464870
2,-0.765899,-0.018803,-0.193790,4.000141,1.366226,3.391072,-0.939656,0.464870
3,-0.603555,-0.229137,0.799097,4.000141,1.053554,3.579267,-0.445671,-2.151139
4,-1.090587,-1.072366,-0.690234,-1.130398,-1.268682,0.756347,-0.939656,0.464870


In [7]:
df.select_dtypes(include=['object']).columns

Index(['person_home_ownership', 'loan_intent', 'cb_person_default_on_file'], dtype='object')

In [8]:
df['loan_intent'].value_counts()

loan_intent
EDUCATION            6288
MEDICAL              5891
VENTURE              5553
PERSONAL             5365
DEBTCONSOLIDATION    5064
HOMEIMPROVEMENT      3510
Name: count, dtype: int64

In [9]:
df['cb_person_default_on_file'].value_counts()

cb_person_default_on_file
N    26043
Y     5628
Name: count, dtype: int64

In [10]:
person_home = pd.get_dummies(df['person_home_ownership'],drop_first=True).astype(np.int8)
loans_intent = pd.get_dummies(df['loan_intent'], drop_first=True).astype(np.int8)
cb_person_default = df['cb_person_default_on_file'].map({'N':0, 'Y':1}).astype(np.int8)
df_oth = pd.concat([X_scaled_df, person_home, loans_intent, cb_person_default, df['loan_status']], axis=1)
df_oth.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,cb_person_codes,OTHER,OWN,RENT,EDUCATION,HOMEIMPROVEMENT,MEDICAL,PERSONAL,VENTURE,cb_person_default_on_file,loan_status
0,-1.090587,-1.078051,0.054432,-1.367192,0.034115,-0.655113,-0.939656,0.464870,0,1,0,1,0,0,0,0,0,0
1,-0.441211,-1.078051,-0.938456,-0.656810,0.597575,3.767461,-0.692664,0.464870,0,0,0,0,0,1,0,0,0,1
2,-0.765899,-0.018803,-0.193790,4.000141,1.366226,3.391072,-0.939656,0.464870,0,0,1,0,0,1,0,0,0,1
3,-0.603555,-0.229137,0.799097,4.000141,1.053554,3.579267,-0.445671,-2.151139,0,0,1,0,0,1,0,0,1,1
4,-1.090587,-1.072366,-0.690234,-1.130398,-1.268682,0.756347,-0.939656,0.464870,0,1,0,0,0,0,0,1,0,1


In [11]:
df_oth['loan_status'].value_counts()

loan_status
0    24846
1     6825
Name: count, dtype: int64

In [12]:
df_oth.isna().sum()

person_age                    0
person_income                 0
person_emp_length             0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_cred_hist_length    0
cb_person_codes               0
OTHER                         0
OWN                           0
RENT                          0
EDUCATION                     0
HOMEIMPROVEMENT               0
MEDICAL                       0
PERSONAL                      0
VENTURE                       0
cb_person_default_on_file     0
loan_status                   0
dtype: int64

In [13]:
from imblearn.over_sampling import SMOTE
smote = SMOTE()

In [14]:
X1 = df_oth.drop('loan_status',axis=1)
y1 = df_oth['loan_status']

In [15]:
X_smote,y_smote=smote.fit_resample(X1,y1) # type: ignore

In [16]:
X_smote.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,cb_person_codes,OTHER,OWN,RENT,EDUCATION,HOMEIMPROVEMENT,MEDICAL,PERSONAL,VENTURE,cb_person_default_on_file
0,-1.090587,-1.078051,0.054432,-1.367192,0.034115,-0.655113,-0.939656,0.464870,0,1,0,1,0,0,0,0,0
1,-0.441211,-1.078051,-0.938456,-0.656810,0.597575,3.767461,-0.692664,0.464870,0,0,0,0,0,1,0,0,0
2,-0.765899,-0.018803,-0.193790,4.000141,1.366226,3.391072,-0.939656,0.464870,0,0,1,0,0,1,0,0,0
3,-0.603555,-0.229137,0.799097,4.000141,1.053554,3.579267,-0.445671,-2.151139,0,0,1,0,0,1,0,0,1
4,-1.090587,-1.072366,-0.690234,-1.130398,-1.268682,0.756347,-0.939656,0.464870,0,1,0,0,0,0,0,1,0


In [17]:
y_smote.value_counts()

loan_status
0    24846
1    24846
Name: count, dtype: int64

In [18]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X_smote,y_smote,test_size=0.2,random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((39753, 17), (9939, 17), (39753,), (9939,))

In [19]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [20]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [21]:
xgb_model = XGBClassifier(eval_metric='logloss')

In [22]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.7, 0.9], # Subsample ratio of the training instance
    'colsample_bytree': [0.7, 1.0]
}

In [23]:
n_splits = 5
skfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [24]:
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='accuracy', # Choose your main evaluation metric
    cv=skfold,          # Use the defined K-Fold object
    verbose=1,         # Controls the verbosity: 1 is standard
    n_jobs=-1          # Use all available cores for parallel processing
)

In [25]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'colsample_bytree': [0.7, 1.0], 'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200, ...], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [26]:
grid_search.best_params_

{'colsample_bytree': 0.7,
 'learning_rate': 0.3,
 'max_depth': 7,
 'n_estimators': 200,
 'subsample': 0.9}

In [27]:
best_model = grid_search.best_estimator_

In [28]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.97      0.95      4995
           1       0.97      0.92      0.95      4944

    accuracy                           0.95      9939
   macro avg       0.95      0.95      0.95      9939
weighted avg       0.95      0.95      0.95      9939



In [ ]:
X_test

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,cb_person_codes,OTHER,OWN,RENT,EDUCATION,HOMEIMPROVEMENT,MEDICAL,PERSONAL,VENTURE,cb_person_default_on_file
24808,0.857541,0.369651,0.054432,0.842886,1.776607,0.003568,0.789289,-2.151139,0,0,0,0,0,0,0,1,1
9935,-0.765899,-0.009329,-0.938456,0.558733,0.034115,0.285860,-0.445671,0.464870,0,0,0,1,0,0,0,0,0
14054,-0.928243,-0.595534,-0.193790,-1.209329,-0.864815,-1.031502,-0.939656,0.464870,0,0,1,0,0,0,0,0,0
147,-0.278867,2.719324,1.295541,1.947926,0.258848,-0.655113,-0.445671,0.464870,0,1,0,0,0,0,1,0,0
4070,-0.928243,-0.344726,-1.186678,-0.814672,-0.415350,-0.749210,-0.939656,0.464870,0,0,1,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6912,-0.278867,0.263537,0.302653,-0.420016,-0.125478,-0.749210,-0.939656,0.464870,0,0,1,0,0,0,0,1,0
24296,0.208165,0.255957,2.040206,0.053573,-1.021151,-0.372821,0.789289,0.464870,0,0,0,1,0,0,0,0,0
29729,3.617388,-0.123023,0.799097,-1.209329,-1.154688,-1.313794,2.271242,0.464870,0,0,0,0,0,0,0,0,0
30345,2.643324,0.257852,2.784872,-0.262153,-0.213416,-0.655113,2.518234,0.464870,0,0,0,0,0,1,0,0,0


In [ ]:
pd.DataFrame({"x_test_indices":X_test.index,"y_test":y_test,"y_pred":y_pred})

,x_test_indices,y_test,y_pred
24808,24808,0,0
9935,9935,0,0
14054,14054,0,0
147,147,0,0
4070,4070,1,1
...,...,...,...
6912,6912,0,0
24296,24296,0,0
29729,29729,0,0
30345,30345,0,0


In [ ]:
res_df = df.merge(pd.DataFrame({"x_test_indices":X_test.index,"y_test":y_test,"y_pred":y_pred}), left_index=True, right_on="x_test_indices", how="inner")

In [ ]:
res_df.isna().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
cb_person_codes               0
x_test_indices                0
y_test                        0
y_pred                        0
dtype: int64

In [ ]:
res_df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,cb_person_codes,x_test_indices,y_test,y_pred
1,25,9600,MORTGAGE,1.0,MEDICAL,5500,12.87,1,0.57,N,3,1,1,1,1
4,21,9900,OWN,2.0,VENTURE,2500,7.14,1,0.25,N,2,1,4,1,1
6,24,78956,RENT,5.0,MEDICAL,35000,11.11,1,0.44,N,4,1,6,1,1
7,24,83000,RENT,8.0,PERSONAL,35000,8.90,1,0.42,N,2,1,7,1,1
38,23,71500,RENT,3.0,DEBTCONSOLIDATION,30000,10.99,1,0.42,N,4,1,38,1,1


In [ ]:
res_df.to_csv('../datasets/credit_risk_model_results.csv', index=False)

In [34]:
import mlflow
import mlflow.xgboost
from mlflow.models.signature import infer_signature

In [35]:
mlflow.set_tracking_uri('http://127.0.0.1:5000')

In [36]:
mlflow.set_experiment("XGBoost_Hyperparameter_Tuning")

2025/10/28 23:20:22 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Hyperparameter_Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/768468838198615142', creation_time=1761673822436, experiment_id='768468838198615142', last_update_time=1761673822436, lifecycle_stage='active', name='XGBoost_Hyperparameter_Tuning', tags={}>

In [37]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.7, 0.9], # Subsample ratio of the training instance
    'colsample_bytree': [0.7, 1.0]
}
n_splits = 5
skfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [38]:
from sklearn.metrics import accuracy_score
import json
import yaml
import mlflow.xgboost
with mlflow.start_run(run_name="XGBoost_Hyperparameter_Tuning") as run:
    run_id = run.info.run_id
    grid_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=param_grid,
        scoring='accuracy', # Choose your main evaluation metric
        cv=skfold,          # Use the defined K-Fold object
        verbose=1,         # Controls the verbosity: 1 is standard
        n_jobs=-1          # Use all available cores for parallel processing
    )
    grid_search.fit(X_train, y_train)
    best_params = grid_search.best_params_
    best_model = grid_search.best_estimator_
    mlflow.log_dict(param_grid, "search_space/full_param_grid.json")
    mlflow.log_params(best_params)
    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy) # type: ignore
    mlflow.log_metric("f1_score_weighted", report['weighted avg']['f1-score']) # type: ignore
    mlflow.log_metric("precision_weighted", report['weighted avg']['precision']) # type: ignore

    # Save to JSON file
    with open("best_params.json", "w") as f:
        json.dump(best_params, f, indent=4)
    mlflow.log_artifact("best_params.json")

    # Save to YAML file
    with open("best_params.yaml", "w") as f:
        yaml.dump(best_params, f, default_flow_style=False)
    mlflow.log_artifact("best_params.yaml")

    np.savetxt("confusion_matrix.txt", cm, fmt="%d")
    mlflow.log_artifact("confusion_matrix.txt")

    with open("classification_report.json", "w") as f:
        json.dump(report, f, indent=4)
    mlflow.log_artifact("classification_report.json")

    signature = infer_signature(X_train, best_model.predict(X_train))

    mlflow.xgboost.log_model( # type: ignore
        xgb_model=best_model,
        artifact_path="xgboost_model",
        signature=signature,
        # Register the model in the MLflow Model Registry
        registered_model_name="BestXGBoostClassifier" 
    )

print(f"\nMLflow Run ID: {run_id}")
print(f"Best Parameters: {best_params}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits


c:\Users\Ann\Desktop\workspace\Model_risk_analytics\pd_model\vpd\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/10/28 23:29:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\Ann\Desktop\workspace\Model_risk_analytics\pd_model\vpd\Lib\site-packages\xgboost\sklearn

🏃 View run XGBoost_Hyperparameter_Tuning at: http://127.0.0.1:5000/#/experiments/768468838198615142/runs/2fbb34dde6d842629c7613ef481a720e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/768468838198615142

MLflow Run ID: 2fbb34dde6d842629c7613ef481a720e
Best Parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.3, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.9}
